# Benchmark baseline vs ML vs time-series

Objetivo: comparar un baseline determinista contra futuros challengers ML y forecasting sin tocar brokers, credenciales ni datos live.

Criterio de exito de este notebook inicial: producir metricas reproducibles sobre datos sinteticos y dejar puntos claros para reemplazar el dataset por OHLCV versionado.

## Setup reproducible

Este notebook usa solo librerias estandar y los modulos locales de `src/trading_ai`. Cuando se instalen dependencias de Fase A, las celdas marcadas como extension pueden incorporar pandas, scikit-learn, MLflow, Evidently, LightGBM, XGBoost, TimesFM o Chronos.

In [ ]:
import math
import random
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

# Imports below depend on the sys.path insert above (local src checkout), so they
# intentionally come after it.
from trading_ai.research.metrics import (  # noqa: E402
    annualized_sharpe,
    cumulative_return,
    max_drawdown,
    volatility_target_weight,
)
from trading_ai.risk.policy import RiskLimits, evaluate_risk_state  # noqa: E402

SEED = 42
random.seed(SEED)
PERIODS_PER_YEAR = 252

## Plan del benchmark

1. Generar una serie sintetica de retornos diarios con regimenes simples.
2. Evaluar un baseline de momentum + volatility targeting.
3. Registrar metricas minimas: retorno acumulado, Sharpe anualizado, max drawdown y peso objetivo.
4. Ejecutar el risk gate en modo paper y live.
5. Sustituir datos sinteticos por OHLCV versionado antes de sacar conclusiones financieras.

In [ ]:
def synthetic_returns(n=504):
    values = []
    for idx in range(n):
        regime_drift = 0.0004 if idx < n // 2 else -0.0001
        regime_vol = 0.009 if idx < n // 2 else 0.014
        values.append(random.gauss(regime_drift, regime_vol))
    return values

returns = synthetic_returns()
print({"observations": len(returns), "seed": SEED, "first_return": round(returns[0], 6)})

## Baseline determinista

La primera comparacion no debe ser un LLM ni un modelo complejo. Este baseline aproxima un sizing por volatilidad objetivo a partir de la volatilidad realizada de la muestra sintetica.

In [ ]:
def sample_std(values):
    mean = sum(values) / len(values)
    return math.sqrt(sum((value - mean) ** 2 for value in values) / (len(values) - 1))

realized_vol = sample_std(returns) * math.sqrt(PERIODS_PER_YEAR)
weight = volatility_target_weight(
    realized_annual_volatility=realized_vol,
    target_annual_volatility=0.12,
    max_leverage=1.0,
)
strategy_returns = [weight * value for value in returns]

metrics = {
    "realized_volatility": round(realized_vol, 4),
    "target_weight": round(weight, 4),
    "cumulative_return": round(cumulative_return(strategy_returns), 4),
    "annualized_sharpe": round(annualized_sharpe(strategy_returns, periods_per_year=PERIODS_PER_YEAR), 4),
    "max_drawdown": round(max_drawdown(strategy_returns), 4),
}
metrics

## Risk gate

El mismo estado que puede ser valido en `paper` debe bloquearse en `live` mientras `configs/permissions.yml` mantenga `live_trading_allowed: false`.

In [ ]:
limits = RiskLimits()
paper_decision = evaluate_risk_state(
    daily_pnl_pct=strategy_returns[-1],
    current_drawdown_pct=max_drawdown(strategy_returns),
    gross_exposure=weight,
    largest_position_weight=weight,
    mode="paper",
    limits=limits,
)
live_decision = evaluate_risk_state(
    daily_pnl_pct=strategy_returns[-1],
    current_drawdown_pct=max_drawdown(strategy_returns),
    gross_exposure=weight,
    largest_position_weight=weight,
    mode="live",
    limits=limits,
)

{
    "paper": paper_decision,
    "live": live_decision,
}

## Extension ML tabular

Cuando existan datos historicos versionados, agregar aqui un modelo tabular simple:

- features: momentum, volatilidad, rango, volumen, regimen y calendario;
- split: walk-forward con embargo si las etiquetas se solapan;
- modelos: logistic/linear baseline, LightGBM o XGBoost;
- tracking: MLflow con hash de dataset, parametros y metricas.

In [ ]:
ml_extension = {
    "status": "not_run",
    "reason": "requires versioned OHLCV dataset and optional ML dependencies",
    "minimum_gate": "must beat deterministic baseline out-of-sample after costs",
}
ml_extension

## Extension forecasting

TimesFM y Chronos deben entrar como challengers de forecasting, no como motor unico de ordenes. Su salida debe convertirse en feature o senal candidata y pasar por los mismos gates de costos, drawdown, exposicion y paper trading.

In [ ]:
forecasting_extension = {
    "status": "not_run",
    "candidates": ["Chronos-Bolt", "Chronos-2", "TimesFM"],
    "minimum_gate": "compare against naive forecast and tabular ML before promotion",
}
forecasting_extension

## Resultado inicial

Este notebook no demuestra rentabilidad. Solo valida que el scaffold puede calcular metricas, aplicar sizing por volatilidad y bloquear live trading por defecto. El siguiente paso tecnico es reemplazar los retornos sinteticos por datos OHLCV versionados y registrar cada corrida en MLflow.